In [65]:
import pulp
import pandas as pd

In [66]:
products_file = "../data/availableProducts.csv"

products_df = pd.read_csv(products_file)
list(products_df["Product name"].unique())

['Pingui',
 'Maxi King',
 'Choco Fresh',
 'Pingui Sachertorte',
 'Paradiso',
 'Milchschnitte',
 'Riegel',
 'Schokobons',
 'Bueno',
 'Happy Hippo',
 'Cards',
 'Überraschung']

In [67]:
number_of_doors = 24
number_of_products = 10

Doors = list(range(1, number_of_doors+1))
Products = list(products_df["Product name"].unique())

max_product_items = 24*2

problem = pulp.LpProblem("AdventCalendarOptimization", pulp.LpMaximize)

In [68]:
number_of_prodcuts_in_door = pulp.LpVariable.dicts("Items_per_door", (Doors, Products), lowBound=0, upBound=1, cat=pulp.LpInteger)

product_used = pulp.LpVariable.dicts("Product_Used", Products, cat=pulp.LpBinary)

In [69]:
problem += pulp.lpSum(product_used[product] for product in Products), "Total product coverage"

In [70]:
for product in Products:
    total_items = pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors)

    problem += total_items <= max_product_items * product_used[product], f"Linking_constraint_{product}"
    problem += total_items >= 1 * product_used[product], f"Linking_constraint_ensure_1_{product}"


In [71]:
MAX_ITEMS_PER_DOOR = 3
MIN_ITEMS_PER_DOOR = 2

for door in Doors:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for product in Products) <= MAX_ITEMS_PER_DOOR

for door in Doors:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for product in Products) >= MIN_ITEMS_PER_DOOR

In [72]:
MAX_PRODUCTS = 20

for p in Products:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors) <= MAX_PRODUCTS

In [73]:
problem.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/leander-merbecks/Documents/git-repositories/projects/ferrerokinderAdventCalendar/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/2515559a80e342fa8b3401c34d35822f-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/2515559a80e342fa8b3401c34d35822f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 89 COLUMNS
At line 2166 RHS
At line 2251 BOUNDS
At line 2552 ENDATA
Problem MODEL has 84 rows, 300 columns and 1464 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 12 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 1 strengthened rows, 0 substitutions
Cgl0004I processed model has 49 rows, 300 columns (300 integer (300 of which binary)) and 913 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers

1

In [74]:
print(f"Optimizer status {pulp.LpStatus[problem.status]}")
print(f"Product coverage {pulp.value(problem.objective)/len(Products)}")

Optimizer status Optimal
Product coverage 1.0


In [75]:
for door in Doors:
    items_in_this_door = []
    total_items = 0

    for product in Products:
        items = pulp.value(number_of_prodcuts_in_door[door][product])

        if items > 0:
            items_in_this_door.append(f"{product}: {int(items)}")
        
        total_items += items
    
    if total_items > 0:
        print(f"Door {door} (total {int(total_items)}): {', '.join(items_in_this_door)}")

Door 1 (total 2): Choco Fresh: 1, Happy Hippo: 1
Door 2 (total 2): Milchschnitte: 1, Riegel: 1
Door 3 (total 2): Maxi King: 1, Schokobons: 1
Door 4 (total 2): Pingui: 1, Choco Fresh: 1
Door 5 (total 2): Riegel: 1, Schokobons: 1
Door 6 (total 2): Pingui: 1, Paradiso: 1
Door 7 (total 2): Pingui: 1, Bueno: 1
Door 8 (total 2): Paradiso: 1, Überraschung: 1
Door 9 (total 2): Maxi King: 1, Cards: 1
Door 10 (total 2): Pingui Sachertorte: 1, Cards: 1
Door 11 (total 2): Maxi King: 1, Überraschung: 1
Door 12 (total 2): Maxi King: 1, Schokobons: 1
Door 13 (total 2): Pingui: 1, Überraschung: 1
Door 14 (total 2): Cards: 1, Überraschung: 1
Door 15 (total 2): Maxi King: 1, Cards: 1
Door 16 (total 2): Choco Fresh: 1, Happy Hippo: 1
Door 17 (total 2): Pingui: 1, Schokobons: 1
Door 18 (total 2): Paradiso: 1, Schokobons: 1
Door 19 (total 2): Paradiso: 1, Überraschung: 1
Door 20 (total 2): Maxi King: 1, Pingui Sachertorte: 1
Door 21 (total 2): Pingui: 1, Happy Hippo: 1
Door 22 (total 2): Choco Fresh: 1, Ha